# 02 — Daily Orders Historical Data Generation

This notebook generates **6 months of synthetic daily order data** for every restaurant in
`restaurant_dataset.csv`. The generated data simulates realistic patterns:

| Pattern | Effect |
|---|---|
| Weekend surge | +30–50 % orders |
| Holiday peaks | +50–100 % orders |
| Weekday dips | Mon/Tue lower |
| Seasonal trend | Gentle sine wave |
| Restaurant base | Derived from rating, votes, price range |
| Random noise | Gaussian jitter |

**Output:** `data/raw/daily_orders.csv`  
**Schema:** `restaurant_id, date, total_orders, total_revenue, avg_discount, promotion_flag, cancellation_rate, avg_delivery_time`

## 1. Setup & Imports

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
rng = np.random.default_rng(SEED)

# ── Paths ────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"

RESTAURANT_CSV = RAW_DIR / "restaurant_dataset.csv"
CONTEXT_CSV = RAW_DIR / "external_context.csv"
OUTPUT_CSV = RAW_DIR / "daily_orders.csv"

print(f"Project root  : {PROJECT_ROOT}")
print(f"Restaurant CSV: {RESTAURANT_CSV}")
print(f"Output CSV    : {OUTPUT_CSV}")

Project root  : C:\Users\welcome\Desktop\restaurant-sales-forecasting-system\minorProj
Restaurant CSV: C:\Users\welcome\Desktop\restaurant-sales-forecasting-system\minorProj\data\raw\restaurant_dataset.csv
Output CSV    : C:\Users\welcome\Desktop\restaurant-sales-forecasting-system\minorProj\data\raw\daily_orders.csv


## 2. Load Restaurant Data

In [2]:
restaurants = pd.read_csv(RESTAURANT_CSV)

# Rename columns for consistency with our schema
restaurants = restaurants.rename(columns={
    "Restaurant ID": "restaurant_id",
    "Restaurant Name": "restaurant_name",
    "City": "city",
    "Price range": "price_range",
    "Aggregate rating": "aggregate_rating",
    "Votes": "votes",
    "Has Online delivery": "has_online_delivery",
    "Average Cost for two": "avg_cost_for_two",
})

# Convert delivery flag
restaurants["has_online_delivery"] = (
    restaurants["has_online_delivery"].str.strip().str.lower() == "yes"
)

print(f"Total restaurants: {len(restaurants)}")
print(f"\nKey columns:")
print(restaurants[["restaurant_id", "city", "price_range",
                    "aggregate_rating", "votes", "has_online_delivery",
                    "avg_cost_for_two"]].describe().to_string())

Total restaurants: 9551

Key columns:
       restaurant_id  price_range  aggregate_rating         votes  avg_cost_for_two
count   9.551000e+03  9551.000000       9551.000000   9551.000000       9551.000000
mean    9.051128e+06     1.804837          2.666370    156.909748       1199.210763
std     8.791521e+06     0.905609          1.516378    430.169145      16121.183073
min     5.300000e+01     1.000000          0.000000      0.000000          0.000000
25%     3.019625e+05     1.000000          2.500000      5.000000        250.000000
50%     6.004089e+06     2.000000          3.200000     31.000000        400.000000
75%     1.835229e+07     2.000000          3.700000    131.000000        700.000000
max     1.850065e+07     4.000000          4.900000  10934.000000     800000.000000


## 3. Define Date Range (6 Months)

**2024-07-01 → 2024-12-31** — covers monsoon, post-monsoon, festivals (Navratri, Diwali, Christmas), and winter.

In [3]:
START_DATE = pd.Timestamp("2024-07-01")
END_DATE = pd.Timestamp("2024-12-31")
date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="D")

print(f"Date range: {START_DATE.date()} → {END_DATE.date()}")
print(f"Total days: {len(date_range)}")
print(f"Expected rows: {len(restaurants)} restaurants × {len(date_range)} days "
      f"= {len(restaurants) * len(date_range):,}")

Date range: 2024-07-01 → 2024-12-31
Total days: 184
Expected rows: 9551 restaurants × 184 days = 1,757,384


## 4. Load External Context (Holidays, Weather, Events)

If the external context CSV exists (from Step 8), we use it for holiday / weather / event modifiers.

In [4]:
if CONTEXT_CSV.exists():
    context = pd.read_csv(CONTEXT_CSV, parse_dates=["date"])
    context = context[(context["date"] >= START_DATE) & (context["date"] <= END_DATE)]
    print(f"Loaded external context: {len(context):,} rows")
    HAS_CONTEXT = True
else:
    print("No external_context.csv found — will generate flags inline.")
    HAS_CONTEXT = False

Loaded external context: 25,944 rows


## 5. Compute Restaurant-Specific Base Demand

Each restaurant gets a unique **base daily order count** derived from its attributes:

| Factor | Logic |
|---|---|
| Price range | Budget (1) → higher volume; Premium (4) → lower volume |
| Rating | Higher rating → more orders |
| Votes | Proxy for popularity (log-scaled) |
| Online delivery | Has delivery → 2× base |
| Random offset | Per-restaurant jitter for diversity |

In [5]:
def compute_base_demand(row):
    """Compute base daily order count for a restaurant."""
    # Price range effect: budget restaurants get more delivery orders
    price_factor = {1: 1.4, 2: 1.1, 3: 0.85, 4: 0.65}.get(row["price_range"], 1.0)

    # Rating effect: higher-rated restaurants attract more orders
    rating = row["aggregate_rating"]
    if rating == 0:
        rating_factor = 0.5   # unrated restaurants get fewer orders
    else:
        rating_factor = 0.4 + (rating / 5.0) * 1.0   # 0.4 – 1.4

    # Votes effect (popularity proxy, log-scaled)
    votes = max(row["votes"], 1)
    vote_factor = 0.6 + 0.25 * np.log1p(votes) / np.log1p(1000)  # normalise to ~1000 votes
    vote_factor = min(vote_factor, 1.8)  # cap

    # Online delivery multiplier
    delivery_factor = 2.0 if row["has_online_delivery"] else 0.8

    # Base: ~20 orders/day for an average restaurant
    base = 20.0 * price_factor * rating_factor * vote_factor * delivery_factor

    # Per-restaurant random offset (±20%)
    base *= rng.uniform(0.8, 1.2)

    return max(int(round(base)), 1)


restaurants["base_demand"] = restaurants.apply(compute_base_demand, axis=1)

print("Base demand statistics:")
print(restaurants["base_demand"].describe().to_string())
print(f"\nSample:")
print(restaurants[["restaurant_id", "restaurant_name", "price_range",
                    "aggregate_rating", "votes", "has_online_delivery",
                    "base_demand"]].head(10).to_string())

Base demand statistics:
count    9551.000000
mean       18.301853
std        11.903929
min         3.000000
25%         9.000000
50%        15.000000
75%        23.000000
max        74.000000

Sample:
   restaurant_id                           restaurant_name  price_range  aggregate_rating  votes  has_online_delivery  base_demand
0        6317637                          Le Petit Souffle            3               4.8    314                False           17
1        6304287                          Izakaya Kikufuji            3               4.5    591                False           14
2        6300002                    Heat - Edsa Shangri-La            4               4.4    270                False           12
3        6318506                                      Ooma            4               4.9    365                False           13
4        6314302                               Sambo Kojin            4               4.8    229                False            9
5       18189

## 6. Define Demand Modifiers

In [6]:
# ── Holiday dates (same as external context notebook) ────────
HOLIDAYS_2024 = set(pd.to_datetime([
    "2024-08-15",  # Independence Day
    "2024-08-26",  # Janmashtami
    "2024-10-02",  # Gandhi Jayanti
    "2024-10-12",  # Dussehra
    "2024-12-25",  # Christmas
    "2024-12-31",  # NYE
    # Navratri
    *[f"2024-10-{d:02d}" for d in range(3, 12)],
    # Diwali week
    *[f"2024-11-{d:02d}" for d in range(1, 6)],
]))


def get_day_modifier(dt):
    """Return a multiplier for a given date."""
    dow = dt.dayofweek  # 0=Mon … 6=Sun
    month = dt.month
    day_of_year = dt.dayofyear

    modifier = 1.0

    # ── Weekend surge (+30–50%) ──────────────────────────────
    if dow == 4:       # Friday
        modifier *= 1.20
    elif dow == 5:     # Saturday
        modifier *= rng.uniform(1.30, 1.50)
    elif dow == 6:     # Sunday
        modifier *= rng.uniform(1.30, 1.50)

    # ── Weekday dips ─────────────────────────────────────────
    elif dow == 0:     # Monday
        modifier *= rng.uniform(0.80, 0.90)
    elif dow == 1:     # Tuesday
        modifier *= rng.uniform(0.82, 0.92)

    # ── Holiday peaks (+50–100%) ─────────────────────────────
    if dt in HOLIDAYS_2024:
        modifier *= rng.uniform(1.50, 2.00)

    # ── Seasonal trend (sine wave, peaks in Oct–Nov) ─────────
    seasonal = 1.0 + 0.10 * np.sin(2 * np.pi * (day_of_year - 280) / 365)
    modifier *= seasonal

    # ── Monsoon dip (Jul–Aug: slight decrease) ───────────────
    if month in (7, 8):
        modifier *= rng.uniform(0.88, 0.95)

    return modifier


# Pre-compute modifiers for each date
date_modifiers = {dt: get_day_modifier(dt) for dt in date_range}

# Show a sample
sample_dates = list(date_modifiers.items())[:5]
for dt, mod in sample_dates:
    print(f"{dt.date()} (dow={dt.dayofweek}): modifier = {mod:.3f}")

2024-07-01 (dow=0): modifier = 0.675
2024-07-02 (dow=1): modifier = 0.665
2024-07-03 (dow=2): modifier = 0.793
2024-07-04 (dow=3): modifier = 0.815
2024-07-05 (dow=4): modifier = 0.999


## 7. Generate Daily Orders for All Restaurants

For each restaurant × date, we compute:

- `total_orders` = base_demand × day_modifier × noise
- `total_revenue` = orders × avg_order_value (with variance)
- `avg_discount` = 0–30% (higher on promotions)
- `promotion_flag` = ~15% of days, more on weekends/holidays
- `cancellation_rate` = 2–8% (slightly higher in bad weather / holidays)
- `avg_delivery_time` = 25–55 min (varies by demand)

In [7]:
# ── Load weather data if available ───────────────────────────
weather_lookup = {}
if HAS_CONTEXT:
    for _, row in context.iterrows():
        weather_lookup[(row["date"].date(), row["city"])] = row["weather"]


def generate_restaurant_orders(rest_row, date_range, date_modifiers):
    """Generate daily order records for a single restaurant."""
    rid = rest_row["restaurant_id"]
    base = rest_row["base_demand"]
    city = rest_row["city"]
    avg_cost = rest_row["avg_cost_for_two"]

    # Average order value ≈ cost_for_two / 2, with some variance
    avg_order_value = max(avg_cost / 2.0, 50.0)  # minimum ₹50

    records = []
    for dt in date_range:
        modifier = date_modifiers[dt]

        # ── Gaussian noise (±15%) ────────────────────────────
        noise = rng.normal(1.0, 0.15)
        noise = max(noise, 0.4)  # floor

        # ── Weather modifier ─────────────────────────────────
        weather = weather_lookup.get((dt.date(), city), "Clear")
        if weather == "Rainy":
            weather_mod = rng.uniform(1.10, 1.25)  # more delivery in rain
        elif weather == "Stormy":
            weather_mod = rng.uniform(0.70, 0.85)  # too stormy, less orders
        elif weather == "Snowy":
            weather_mod = rng.uniform(1.05, 1.15)  # snow = stay home = order in
        else:
            weather_mod = 1.0

        # ── Total orders ─────────────────────────────────────
        orders = int(round(base * modifier * noise * weather_mod))
        orders = max(orders, 0)

        # ── Promotion flag (~15%, higher on weekends/holidays) ─
        is_weekend = dt.dayofweek >= 5
        is_holiday = dt in HOLIDAYS_2024
        promo_prob = 0.12
        if is_weekend:
            promo_prob = 0.25
        if is_holiday:
            promo_prob = 0.40
        promotion_flag = bool(rng.random() < promo_prob)

        # ── Average discount ─────────────────────────────────
        if promotion_flag:
            avg_discount = round(rng.uniform(10.0, 30.0), 1)
        else:
            avg_discount = round(rng.uniform(0.0, 8.0), 1)

        # ── Revenue ──────────────────────────────────────────
        discount_factor = 1.0 - (avg_discount / 100.0)
        order_value_noise = rng.normal(1.0, 0.10)
        order_value_noise = max(order_value_noise, 0.5)
        revenue = round(orders * avg_order_value * order_value_noise * discount_factor, 2)
        revenue = max(revenue, 0.0)

        # If no orders, revenue must be 0
        if orders == 0:
            revenue = 0.0

        # ── Cancellation rate (2–8%, higher in bad weather) ──
        cancel_base = rng.uniform(0.02, 0.06)
        if weather in ("Stormy", "Rainy"):
            cancel_base += rng.uniform(0.01, 0.03)
        cancellation_rate = round(min(cancel_base, 0.15), 3)

        # ── Avg delivery time (25–55 min, higher when busy) ──
        demand_pressure = min(orders / max(base, 1), 2.0)
        delivery_time = round(25 + 15 * demand_pressure + rng.normal(0, 3), 1)
        delivery_time = max(delivery_time, 15.0)

        records.append({
            "restaurant_id": rid,
            "date": dt.strftime("%Y-%m-%d"),
            "total_orders": orders,
            "total_revenue": revenue,
            "avg_discount": avg_discount,
            "promotion_flag": promotion_flag,
            "cancellation_rate": cancellation_rate,
            "avg_delivery_time": delivery_time,
        })

    return records


print("Generation function defined. Ready to run.")

Generation function defined. Ready to run.


## 8. Run Generation (All Restaurants)

In [8]:
all_records = []
total = len(restaurants)

for idx, (_, rest_row) in enumerate(restaurants.iterrows()):
    records = generate_restaurant_orders(rest_row, date_range, date_modifiers)
    all_records.extend(records)

    if (idx + 1) % 2000 == 0 or (idx + 1) == total:
        print(f"  Processed {idx + 1:,} / {total:,} restaurants "
              f"({(idx + 1) / total * 100:.0f}%) — "
              f"{len(all_records):,} rows generated")

df = pd.DataFrame(all_records)
print(f"\n✅ Generation complete: {df.shape}")

  Processed 2,000 / 9,551 restaurants (21%) — 368,000 rows generated


  Processed 4,000 / 9,551 restaurants (42%) — 736,000 rows generated


  Processed 6,000 / 9,551 restaurants (63%) — 1,104,000 rows generated


  Processed 8,000 / 9,551 restaurants (84%) — 1,472,000 rows generated


  Processed 9,551 / 9,551 restaurants (100%) — 1,757,384 rows generated



✅ Generation complete: (1757384, 8)


## 9. Validation & Summary

In [9]:
# ── Schema checks ────────────────────────────────────────────
expected_cols = ["restaurant_id", "date", "total_orders", "total_revenue",
                 "avg_discount", "promotion_flag", "cancellation_rate",
                 "avg_delivery_time"]
assert list(df.columns) == expected_cols, f"Column mismatch: {list(df.columns)}"
assert df.isnull().sum().sum() == 0, "Null values found!"
assert (df["total_orders"] >= 0).all(), "Negative orders!"
assert (df["total_revenue"] >= 0).all(), "Negative revenue!"
assert (df["avg_discount"].between(0, 100)).all(), "Discount out of range!"
assert (df["cancellation_rate"].between(0, 1)).all(), "Cancellation rate out of range!"
assert (df["avg_delivery_time"] >= 0).all(), "Negative delivery time!"

# ── Revenue = 0 when orders = 0 ─────────────────────────────
zero_orders = df[df["total_orders"] == 0]
if len(zero_orders) > 0:
    assert (zero_orders["total_revenue"] == 0).all(), "Revenue > 0 when orders = 0!"

# ── Unique restaurants check ─────────────────────────────────
assert df["restaurant_id"].nunique() == len(restaurants), "Missing restaurants!"

# ── Summary ──────────────────────────────────────────────────
print("=" * 60)
print("        DAILY ORDERS DATASET — SUMMARY")
print("=" * 60)
print(f"Shape            : {df.shape}")
print(f"Date range       : {df['date'].min()} → {df['date'].max()}")
print(f"Restaurants      : {df['restaurant_id'].nunique():,}")
print(f"Rows per rest.   : {len(date_range)}")
print(f"Null values      : {df.isnull().sum().sum()}")
print(f"\nOrders/day stats:")
print(df["total_orders"].describe().to_string())
print(f"\nRevenue/day stats:")
print(df["total_revenue"].describe().to_string())
print(f"\nPromotion %      : {df['promotion_flag'].mean() * 100:.1f}%")
print(f"Avg discount     : {df['avg_discount'].mean():.1f}%")
print(f"Avg cancel rate  : {df['cancellation_rate'].mean():.3f}")
print(f"Avg delivery min : {df['avg_delivery_time'].mean():.1f}")

print("\n✅ All validation checks passed!")

        DAILY ORDERS DATASET — SUMMARY
Shape            : (1757384, 8)


Date range       : 2024-07-01 → 2024-12-31
Restaurants      : 9,551
Rows per rest.   : 184
Null values      : 0

Orders/day stats:


count    1.757384e+06
mean     2.170559e+01
std      1.704231e+01
min      1.000000e+00
25%      1.000000e+01
50%      1.600000e+01
75%      2.800000e+01
max      2.730000e+02

Revenue/day stats:
count    1.757384e+06
mean     1.064654e+04
std      1.332227e+05
min      7.567000e+01
25%      1.315910e+03
50%      3.326715e+03
75%      7.958792e+03
max      1.828625e+07

Promotion %      : 18.4%
Avg discount     : 6.9%
Avg cancel rate  : 0.047
Avg delivery min : 42.5

✅ All validation checks passed!


## 10. Weekend & Holiday Pattern Verification

In [10]:
df["_date"] = pd.to_datetime(df["date"])
df["_dow"] = df["_date"].dt.dayofweek
df["_is_weekend"] = df["_dow"] >= 5
df["_is_holiday"] = df["_date"].isin(HOLIDAYS_2024)

weekday_avg = df[~df["_is_weekend"] & ~df["_is_holiday"]]["total_orders"].mean()
weekend_avg = df[df["_is_weekend"] & ~df["_is_holiday"]]["total_orders"].mean()
holiday_avg = df[df["_is_holiday"]]["total_orders"].mean()

print(f"Avg orders — Weekday : {weekday_avg:.1f}")
print(f"Avg orders — Weekend : {weekend_avg:.1f}  "
      f"(+{(weekend_avg / weekday_avg - 1) * 100:.0f}% vs weekday)")
print(f"Avg orders — Holiday : {holiday_avg:.1f}  "
      f"(+{(holiday_avg / weekday_avg - 1) * 100:.0f}% vs weekday)")

# Clean up temp columns
df.drop(columns=["_date", "_dow", "_is_weekend", "_is_holiday"], inplace=True)

print("\n✅ Patterns verified — weekend and holiday surges look realistic!")

Avg orders — Weekday : 17.7
Avg orders — Weekend : 25.7  (+45% vs weekday)
Avg orders — Holiday : 35.4  (+100% vs weekday)

✅ Patterns verified — weekend and holiday surges look realistic!


## 11. Save to CSV

In [11]:
df.to_csv(OUTPUT_CSV, index=False)

file_size_mb = OUTPUT_CSV.stat().st_size / (1024 * 1024)
print(f"✅ Saved to: {OUTPUT_CSV}")
print(f"   File size: {file_size_mb:.1f} MB")
print(f"   Rows     : {len(df):,}")
print(f"   Columns  : {list(df.columns)}")

✅ Saved to: C:\Users\welcome\Desktop\restaurant-sales-forecasting-system\minorProj\data\raw\daily_orders.csv
   File size: 85.8 MB
   Rows     : 1,757,384
   Columns  : ['restaurant_id', 'date', 'total_orders', 'total_revenue', 'avg_discount', 'promotion_flag', 'cancellation_rate', 'avg_delivery_time']


In [12]:
# ── Preview ──────────────────────────────────────────────────
preview = pd.read_csv(OUTPUT_CSV)
print("Preview (first 10 rows):")
preview.head(10)

Preview (first 10 rows):


,restaurant_id,date,total_orders,total_revenue,avg_discount,promotion_flag,cancellation_rate,avg_delivery_time
0,6317637,2024-07-01,13,6649.26,3.1,False,0.046,35.4
1,6317637,2024-07-02,15,7636.46,6.7,False,0.086,34.2
2,6317637,2024-07-03,14,5930.51,3.5,False,0.023,42.2
3,6317637,2024-07-04,14,6032.38,24.7,True,0.038,35.5
4,6317637,2024-07-05,20,12043.87,0.7,False,0.074,41.3
5,6317637,2024-07-06,22,10106.00,12.4,True,0.048,46.2
6,6317637,2024-07-07,18,6917.42,5.3,False,0.041,40.3
7,6317637,2024-07-08,14,7373.74,1.7,False,0.029,33.7
8,6317637,2024-07-09,14,5331.89,13.8,True,0.021,35.9
9,6317637,2024-07-10,18,7418.69,22.2,True,0.064,37.6
